# Paso 4 — Integración de Datos (Versión Python / Jupyter)

Este notebook hace **lo mismo** que `cargar_dw.py` + los scripts `sql/01` a `sql/06`, pero la limpieza y la transformación se hacen en **Python con la librería pandas** en lugar de hacerlas con SQL.

Qué hace, en orden:

1. **Extracción:** lee `Matches.csv`.
2. **Calidad de datos:** revisa vacíos, valores imposibles, nombres mal escritos, duplicados, etc.
3. **Limpieza:** corrige lo que se puede corregir y descarta lo que no sirve.
4. **Transformación:** arma las 4 dimensiones y la tabla de hechos (el modelo estrella del Paso 3).
5. **Carga:** crea la base `dw_competencia_futbol` en MySQL y guarda las tablas.
6. **Verificación:** compara lo que quedó en MySQL contra el archivo original.

**Cómo se usa:** cada bloque gris es una *celda de código*. Se ejecuta haciendo clic en ella y apretando **Shift + Enter**. Hay que ejecutarlas **en orden, de arriba hacia abajo** (o usar el menú *Run → Run All Cells*). El resultado de cada celda aparece justo debajo.

## 0. Preparación

### 0.1 Importar las librerías

Una *librería* es código que ya escribió otra persona y que nosotros usamos (como `#include <iostream>` en C++).

* `pandas`: sirve para trabajar con tablas (en pandas una tabla se llama **DataFrame**).
* `mysql.connector`: para conectarse a MySQL y ejecutar SQL.
* `sqlalchemy`: lo usa pandas para guardar un DataFrame entero como tabla de MySQL.
* `os` y `getpass`: para pedir la contraseña de MySQL sin escribirla en el notebook.

In [ ]:
import os
from getpass import getpass

import pandas as pd
import mysql.connector
from sqlalchemy import create_engine, URL

### 0.2 Configuración

Acá están las rutas de los archivos y los datos de conexión. **Si en tu PC algo es distinto, se cambia solo acá.**

* `Matches.csv` tiene que estar en la carpeta `datos/` (al lado de este notebook).
* `RECREAR_TABLAS`: si es `False` (por defecto), **no borra** la base de datos si ya existe, permitiendo que quede guardada permanentemente en tu computadora. Si alguna vez necesitas empezar de cero, cámbialo a `True`.

In [ ]:
RUTA_CSV = "datos/Matches.csv"
RUTA_DDL = "../paso-3/entrega_modelo_logico.sql"   # el modelo lógico del Paso 3
RUTA_MAPEO_DIVISIONES = "mapeos/mapeo_divisiones.csv"
RUTA_MAPEO_EQUIPOS = "mapeos/mapeo_equipos.csv"

SERVIDOR = "localhost"
PUERTO = 3306
USUARIO = "root"
BASE_DE_DATOS = "dw_competencia_futbol"

# Control de persistencia:
# False = Conserva la base de datos existente en disco (evita borrar y recrear al reiniciar kernel)
# True  = Borra y vuelve a crear la base y las tablas desde cero (DROP TABLE)
RECREAR_TABLAS = False


---
## 1. Extracción: leer el archivo

`pd.read_csv` lee el CSV y lo guarda en un DataFrame (una tabla en memoria).

Lo leemos **todo como texto** (`dtype=str`) y dejando los vacíos como texto vacío (`keep_default_na=False`). Es el equivalente a la tabla de *staging* `STG_PARTIDOS` de la versión SQL: primero miramos el dato tal cual viene y después lo convertimos.

In [ ]:
partidos = pd.read_csv(RUTA_CSV, dtype=str, keep_default_na=False)

print("Filas leídas:", len(partidos))
print("Columnas:", len(partidos.columns))

`.head()` muestra las primeras 5 filas. Sirve para ver cómo vienen los datos.

In [ ]:
partidos.head()

El archivo trae 48 columnas, pero el modelo del Paso 3 solo usa 13 (las cuotas de apuestas, los tiros, los córners, etc. se descartaron en el Paso 3). Nos quedamos solo con esas.

In [ ]:
columnas_usadas = [
    "Division", "MatchDate", "HomeTeam", "AwayTeam",
    "HomeElo", "AwayElo", "FTHome", "FTAway", "FTResult",
    "HomeYellow", "AwayYellow", "HomeRed", "AwayRed",
]
partidos = partidos[columnas_usadas]
partidos.head()

### 1.1 Leer las tablas de mapeo

Son dos CSV chiquitos que armamos nosotros (son los mismos que usa la versión SQL):

* `mapeo_divisiones.csv`: las **15 ligas del alcance**, con su nombre y su país.
* `mapeo_equipos.csv`: los clubes que la fuente escribe de dos formas distintas, y el nombre que vamos a usar.

In [ ]:
mapeo_divisiones = pd.read_csv(RUTA_MAPEO_DIVISIONES)
mapeo_equipos = pd.read_csv(RUTA_MAPEO_EQUIPOS)

mapeo_divisiones

In [ ]:
mapeo_equipos

---
## 2. Calidad de datos

Antes de cargar nada, revisamos qué problemas tiene el archivo. Cada celda es un chequeo (son los mismos de `sql/02_calidad.sql`).

**Cómo se lee el código:**

* `partidos["HomeTeam"]` es la columna `HomeTeam` (se llama *Series*).
* `partidos["FTHome"] == ""` da, para cada fila, `True` si está vacía o `False` si no.
* `.sum()` suma esos `True` (cada `True` vale 1), o sea, **cuenta** cuántas filas cumplen.
* `.str.strip()` le saca los espacios del principio y del final a cada texto (como `TRIM` en SQL).

### 2.1 Vacíos en las columnas clave

In [ ]:
for columna in ["MatchDate", "Division", "HomeTeam", "AwayTeam", "FTHome", "FTAway", "FTResult"]:
    vacios = (partidos[columna].str.strip() == "").sum()
    print(columna, "->", vacios, "vacíos")

### 2.2 Vacíos en Elo y tarjetas (archivo completo)

In [ ]:
total = len(partidos)
for columna in ["HomeElo", "AwayElo", "HomeYellow", "HomeRed"]:
    vacios = (partidos[columna].str.strip() == "").sum()
    porcentaje = round(100 * vacios / total, 1)
    print(columna, "->", vacios, "vacíos (", porcentaje, "% )")

### 2.3 Fechas: formato y rango

Las fechas tienen que tener el formato `AAAA-MM-DD`. `.str.match(...)` revisa ese patrón y el `~` significa "NO" (invierte `True` y `False`).

In [ ]:
formato_ok = partidos["MatchDate"].str.match(r"^\d{4}-\d{2}-\d{2}$")
print("Fechas con formato inválido:", (~formato_ok).sum())
print("Primera fecha:", partidos["MatchDate"].min())
print("Última fecha:", partidos["MatchDate"].max())

### 2.4 Resultado: dominio y coherencia con los goles

`FTResult` solo puede ser `H` (gana el local), `D` (empate) o `A` (gana el visitante), y tiene que coincidir con los goles.

`pd.to_numeric(..., errors="coerce")` convierte texto a número; si no puede (por ejemplo, un vacío) deja `NaN`, que es como pandas marca un dato faltante (el `NULL` de SQL).

In [ ]:
resultados_validos = partidos["FTResult"].isin(["H", "D", "A"]) | (partidos["FTResult"] == "")
print("Resultados fuera de H / D / A:", (~resultados_validos).sum())

goles_local = pd.to_numeric(partidos["FTHome"], errors="coerce")
goles_visitante = pd.to_numeric(partidos["FTAway"], errors="coerce")

resultado_calculado = pd.Series("D", index=partidos.index)
resultado_calculado[goles_local > goles_visitante] = "H"
resultado_calculado[goles_local < goles_visitante] = "A"

tiene_goles = goles_local.notna() & goles_visitante.notna()
inconsistentes = tiene_goles & (partidos["FTResult"] != resultado_calculado)
print("Resultados que no coinciden con los goles:", inconsistentes.sum())

### 2.5 Valores fuera de rango

Límites razonables: goles ≤ 15, amarillas ≤ 15 por equipo, rojas ≤ 5 por equipo (con más de 5 expulsados el partido se suspende) y Elo entre 800 y 2500.

En pandas, `|` significa "o" y `&` significa "y" (como `||` y `&&` en C++).

In [ ]:
def numero(columna):
    return pd.to_numeric(partidos[columna], errors="coerce")

fuera_de_rango = (
    (numero("FTHome") > 15) | (numero("FTAway") > 15)
    | (numero("HomeYellow") > 15) | (numero("AwayYellow") > 15)
    | (numero("HomeRed") > 5) | (numero("AwayRed") > 5)
    | (numero("HomeElo") < 800) | (numero("HomeElo") > 2500)
    | (numero("AwayElo") < 800) | (numero("AwayElo") > 2500)
)
print("Filas con algún valor fuera de rango:", fuera_de_rango.sum())
partidos[fuera_de_rango]

### 2.6 Nombres de equipo con espacios sobrantes

Juntamos locales y visitantes en una sola lista (`pd.concat`) y buscamos los nombres que cambian al sacarles los espacios. `.value_counts()` cuenta cuántas veces aparece cada uno.

In [ ]:
equipos = pd.concat([partidos["HomeTeam"], partidos["AwayTeam"]])
con_espacios = equipos[equipos != equipos.str.strip()]
con_espacios.value_counts()

### 2.7 Mismo club escrito de dos formas

Por ejemplo, la fuente escribe `Nottm Forest` hasta 2023 y `Nott'm Forest` desde 2024. Sin unificar, el DW tendría dos equipos distintos. Contamos cuántas veces aparece cada variante de la tabla de mapeo.

In [ ]:
equipos_sin_espacios = equipos.str.strip()
for nombre in mapeo_equipos["nombre_origen"]:
    apariciones = (equipos_sin_espacios == nombre).sum()
    print(nombre, "->", apariciones, "apariciones")

### 2.8 Partidos dentro y fuera del alcance (15 ligas europeas)

`.isin(lista)` da `True` si el valor está en la lista.

In [ ]:
en_alcance = partidos["Division"].isin(mapeo_divisiones["codigo_division"])
print("Dentro del alcance:", en_alcance.sum(), "partidos")
print("Fuera del alcance:", (~en_alcance).sum(), "partidos")
print("Ligas fuera del alcance:", sorted(partidos.loc[~en_alcance, "Division"].unique()))

### 2.9 Vacíos de Elo y tarjetas dentro del alcance

In [ ]:
del_alcance = partidos[en_alcance]
sin_elo = (del_alcance["HomeElo"] == "") | (del_alcance["AwayElo"] == "")
sin_tarjetas = del_alcance["HomeYellow"] == ""
print("Partidos en el alcance:", len(del_alcance))
print("Sin Elo:", sin_elo.sum(), "(", round(100 * sin_elo.mean(), 1), "% )")
print("Sin tarjetas:", sin_tarjetas.sum(), "(", round(100 * sin_tarjetas.mean(), 1), "% )")

### 2.10 Duplicados y partidos contra sí mismo

In [ ]:
clave = pd.DataFrame({
    "fecha": partidos["MatchDate"],
    "local": partidos["HomeTeam"].str.strip(),
    "visitante": partidos["AwayTeam"].str.strip(),
})
print("Partidos duplicados (misma fecha, local y visitante):", clave.duplicated().sum())
print("Partidos donde local y visitante son el mismo equipo:", (clave["local"] == clave["visitante"]).sum())

---
## 3. Limpieza

Reglas (las mismas que `sql/03_limpieza.sql`):

1. Se descartan los partidos de ligas **fuera del alcance**.
2. Se sacan los espacios sobrantes de los nombres y se unifican las variantes con la tabla de mapeo.
3. Se convierten los tipos: la fecha a fecha y los números a número.
4. Los valores fuera de rango se reemplazan por vacío (`NaN`) y el partido se conserva.
5. Tarjetas y Elo vacíos **quedan vacíos**: no se rellenan con 0, porque eso bajaría los promedios.
6. Se descartan los partidos sin fecha, sin equipos, sin goles, duplicados o de un equipo contra sí mismo.
7. El resultado se recalcula a partir de los goles.

Vamos anotando en la lista `descartes` cuántas filas se sacan y por qué.

In [ ]:
filas_archivo = len(partidos)
descartes = []   # lista de (motivo, cantidad de filas)

### 3.1 Alcance: solo las 15 ligas

`partidos[condicion]` se queda con las filas donde la condición es `True`. `.copy()` crea una tabla nueva, así no modificamos la original.

In [ ]:
en_alcance = partidos["Division"].isin(mapeo_divisiones["codigo_division"])
descartes.append(("Liga fuera del alcance", (~en_alcance).sum()))

limpio = partidos[en_alcance].copy()
print("Filas que quedan:", len(limpio))

### 3.2 Textos: espacios, caracteres raros y variantes de nombre

* `.str.strip()` saca los espacios.
* `.str.replace("\x92", "'")` cambia un carácter mal codificado por un apóstrofo.
* `.replace(diccionario)` cambia cada nombre de la izquierda por el de la derecha. Un **diccionario** es una lista de pares `clave: valor` (parecido a un `map` de C++).

In [ ]:
for columna in ["Division", "HomeTeam", "AwayTeam", "FTResult"]:
    limpio[columna] = limpio[columna].str.strip()

limpio["HomeTeam"] = limpio["HomeTeam"].str.replace("\x92", "'")
limpio["AwayTeam"] = limpio["AwayTeam"].str.replace("\x92", "'")

reemplazos = dict(zip(mapeo_equipos["nombre_origen"], mapeo_equipos["nombre_canonico"]))
print("Reemplazos:", reemplazos)

limpio["HomeTeam"] = limpio["HomeTeam"].replace(reemplazos)
limpio["AwayTeam"] = limpio["AwayTeam"].replace(reemplazos)

### 3.3 Conversión de tipos

Hasta ahora todo era texto. Convertimos la fecha con `pd.to_datetime` y los números con `pd.to_numeric`. Lo que está vacío queda como `NaN`.

In [ ]:
limpio["MatchDate"] = pd.to_datetime(limpio["MatchDate"], format="%Y-%m-%d", errors="coerce")

columnas_numericas = ["HomeElo", "AwayElo", "FTHome", "FTAway",
                      "HomeYellow", "AwayYellow", "HomeRed", "AwayRed"]
for columna in columnas_numericas:
    limpio[columna] = pd.to_numeric(limpio[columna], errors="coerce")

limpio.dtypes

### 3.4 Valores fuera de rango → vacío

`limpio.loc[condicion, columna] = None` pone vacío en esa columna, solo en las filas donde la condición es `True`.

In [ ]:
anulados = 0

for columna in ["HomeRed", "AwayRed"]:
    fuera = limpio[columna] > 5
    anulados += fuera.sum()
    limpio.loc[fuera, columna] = None

for columna in ["HomeYellow", "AwayYellow"]:
    fuera = limpio[columna] > 15
    anulados += fuera.sum()
    limpio.loc[fuera, columna] = None

for columna in ["HomeElo", "AwayElo"]:
    fuera = (limpio[columna] < 800) | (limpio[columna] > 2500)
    anulados += fuera.sum()
    limpio.loc[fuera, columna] = None

print("Valores fuera de rango reemplazados por vacío:", anulados)

### 3.5 Descartes por problemas de datos

Cada regla: contamos las filas que no la cumplen, las anotamos en `descartes` y nos quedamos con el resto.

In [ ]:
sin_fecha = limpio["MatchDate"].isna()
descartes.append(("Fecha vacía o inválida", sin_fecha.sum()))
limpio = limpio[~sin_fecha]

sin_equipo = (limpio["HomeTeam"] == "") | (limpio["AwayTeam"] == "")
descartes.append(("Equipo vacío", sin_equipo.sum()))
limpio = limpio[~sin_equipo]

sin_goles = limpio["FTHome"].isna() | limpio["FTAway"].isna()
print("Partidos sin resultado:")
print(limpio[sin_goles])
descartes.append(("Partido sin goles registrados (sin resultado)", sin_goles.sum()))
limpio = limpio[~sin_goles]

mismo_equipo = limpio["HomeTeam"] == limpio["AwayTeam"]
descartes.append(("Local y visitante iguales", mismo_equipo.sum()))
limpio = limpio[~mismo_equipo]

# Si un partido aparece dos veces, se conserva la primera aparición
duplicado = limpio.duplicated(subset=["MatchDate", "HomeTeam", "AwayTeam"], keep="first")
descartes.append(("Partido duplicado", duplicado.sum()))
limpio = limpio[~duplicado]

tabla_descartes = pd.DataFrame(descartes, columns=["motivo", "filas"])
tabla_descartes

### 3.6 Resultado derivado de los goles y tipos finales

* El resultado se recalcula con los goles (así nunca puede quedar incoherente).
* Los goles pasan a entero (`int`).
* Las tarjetas pasan a `Int64`: es un entero que **admite vacíos** (el `int` común de pandas no los admite).

In [ ]:
limpio["resultado"] = "D"
limpio.loc[limpio["FTHome"] > limpio["FTAway"], "resultado"] = "H"
limpio.loc[limpio["FTHome"] < limpio["FTAway"], "resultado"] = "A"

corregidos = (limpio["resultado"] != limpio["FTResult"]).sum()
print("Resultados corregidos:", corregidos)

limpio["FTHome"] = limpio["FTHome"].astype(int)
limpio["FTAway"] = limpio["FTAway"].astype(int)
for columna in ["HomeYellow", "AwayYellow", "HomeRed", "AwayRed"]:
    limpio[columna] = limpio[columna].astype("Int64")

### 3.7 Nombres de columna del modelo y datos de liga y país

Renombramos las columnas a los nombres del Paso 3 y, con `.merge` (que es un `JOIN`), le agregamos a cada partido el nombre de su liga y su país desde la tabla de mapeo.

In [ ]:
limpio = limpio.rename(columns={
    "MatchDate": "fecha",
    "Division": "codigo_division",
    "HomeTeam": "equipo_local",
    "AwayTeam": "equipo_visitante",
    "FTHome": "goles_local",
    "FTAway": "goles_visitante",
    "HomeElo": "elo_local",
    "AwayElo": "elo_visitante",
    "HomeYellow": "amarillas_local",
    "AwayYellow": "amarillas_visitante",
    "HomeRed": "rojas_local",
    "AwayRed": "rojas_visitante",
})
limpio = limpio.drop(columns=["FTResult"])
limpio = limpio.merge(mapeo_divisiones, on="codigo_division")

print("Filas limpias:", len(limpio))
print("Filas descartadas:", tabla_descartes["filas"].sum())
print("Balance (archivo - limpias - descartadas):",
      filas_archivo - len(limpio) - tabla_descartes["filas"].sum())
limpio.head()

---
## 4. Transformación: armar el modelo estrella

Armamos en memoria las mismas 5 tablas del Paso 3. Cada dimensión tiene una **clave subrogada** (`id_...`) que numeramos 1, 2, 3… Después la tabla de hechos usa esos números en lugar de los textos.

### 4.1 DIM_TIEMPO

Una fila por cada fecha con partidos. `.dt.year`, `.dt.month`, etc. sacan partes de la fecha. La **temporada** corta en julio: un partido de marzo de 2022 es de la temporada `2021-2022`.

In [ ]:
dim_tiempo = pd.DataFrame({"fecha": limpio["fecha"].drop_duplicates().sort_values()})

dim_tiempo["anio"] = dim_tiempo["fecha"].dt.year
dim_tiempo["mes"] = dim_tiempo["fecha"].dt.month
dim_tiempo["trimestre"] = dim_tiempo["fecha"].dt.quarter


def calcular_temporada(fecha):
    if fecha.month >= 7:
        return str(fecha.year) + "-" + str(fecha.year + 1)
    else:
        return str(fecha.year - 1) + "-" + str(fecha.year)


dim_tiempo["temporada"] = dim_tiempo["fecha"].apply(calcular_temporada)

nombres_meses = {1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio",
                 7: "Julio", 8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"}
dim_tiempo["nombre_mes"] = dim_tiempo["mes"].map(nombres_meses)

# En pandas el lunes es el día 0 y el domingo el 6
nombres_dias = {0: "Lunes", 1: "Martes", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}
dim_tiempo["dia_semana"] = dim_tiempo["fecha"].dt.dayofweek.map(nombres_dias)

dim_tiempo.insert(0, "id_tiempo", range(1, len(dim_tiempo) + 1))
dim_tiempo = dim_tiempo[["id_tiempo", "fecha", "temporada", "anio", "mes", "nombre_mes", "trimestre", "dia_semana"]]

print("DIM_TIEMPO:", len(dim_tiempo), "filas")
dim_tiempo.head()

### 4.2 DIM_EQUIPO

Todos los equipos que jugaron de local o de visitante, sin repetir y ordenados.

In [ ]:
nombres = pd.concat([limpio["equipo_local"], limpio["equipo_visitante"]]).drop_duplicates().sort_values()
dim_equipo = pd.DataFrame({"nombre_equipo": nombres})
dim_equipo.insert(0, "id_equipo", range(1, len(dim_equipo) + 1))

print("DIM_EQUIPO:", len(dim_equipo), "filas")
dim_equipo.head()

### 4.3 DIM_DIVISION y DIM_PAIS

In [ ]:
dim_division = limpio[["codigo_division", "nombre_liga"]].drop_duplicates().sort_values("codigo_division")
dim_division.insert(0, "id_division", range(1, len(dim_division) + 1))

dim_pais = limpio[["codigo_pais", "nombre_pais"]].drop_duplicates().sort_values("codigo_pais")
dim_pais.insert(0, "id_pais", range(1, len(dim_pais) + 1))

print("DIM_DIVISION:", len(dim_division), "filas")
print("DIM_PAIS:", len(dim_pais), "filas")
dim_division

In [ ]:
dim_pais

### 4.4 FACT_COMPETENCIA

Un registro por partido. Para cada partido buscamos las claves de sus dimensiones con `.merge` (igual que un `JOIN` en SQL):

* la fecha → `id_tiempo`
* el nombre del local → `id_equipo_local`, y el del visitante → `id_equipo_visitante`
* el código de división → `id_division`, y el país → `id_pais`

`DIM_EQUIPO` se usa dos veces (una para el local y otra para el visitante), por eso le cambiamos el nombre a las columnas con `.rename` antes de cada `merge`.

In [ ]:
hechos = limpio.merge(dim_tiempo[["id_tiempo", "fecha"]], on="fecha")

equipo_local = dim_equipo.rename(columns={"id_equipo": "id_equipo_local", "nombre_equipo": "equipo_local"})
hechos = hechos.merge(equipo_local, on="equipo_local")

equipo_visitante = dim_equipo.rename(columns={"id_equipo": "id_equipo_visitante", "nombre_equipo": "equipo_visitante"})
hechos = hechos.merge(equipo_visitante, on="equipo_visitante")

hechos = hechos.merge(dim_division[["id_division", "codigo_division"]], on="codigo_division")
hechos = hechos.merge(dim_pais[["id_pais", "codigo_pais"]], on="codigo_pais")

print("Filas limpias:", len(limpio), "| Filas de hechos:", len(hechos))

**Hechos derivados del resultado:** puntos (3 al que gana, 1 a cada uno si empatan, 0 al que pierde) y las banderas `victoria_local`, `empate`, `victoria_visitante` (1 = sí, 0 = no).

In [ ]:
hechos["puntos_local"] = hechos["resultado"].map({"H": 3, "D": 1, "A": 0})
hechos["puntos_visitante"] = hechos["resultado"].map({"H": 0, "D": 1, "A": 3})
hechos["victoria_local"] = (hechos["resultado"] == "H").astype(int)
hechos["empate"] = (hechos["resultado"] == "D").astype(int)
hechos["victoria_visitante"] = (hechos["resultado"] == "A").astype(int)

# Nos quedamos solo con las columnas de FACT_COMPETENCIA, en el orden del Paso 3
hechos = hechos[[
    "id_tiempo", "id_equipo_local", "id_equipo_visitante", "id_division", "id_pais",
    "goles_local", "goles_visitante", "resultado",
    "elo_local", "elo_visitante",
    "amarillas_local", "amarillas_visitante", "rojas_local", "rojas_visitante",
    "puntos_local", "puntos_visitante", "victoria_local", "empate", "victoria_visitante",
]]
hechos.head()

---
## 5. Carga en MySQL

### 5.1 Contraseña

Si la variable de entorno `MYSQL_PASSWORD` existe, se usa esa. Si no, aparece un cuadro para escribir la contraseña (no queda guardada en el notebook).

In [ ]:
contrasena = os.environ.get("MYSQL_PASSWORD") or getpass("Contraseña de MySQL: ")

### 5.2 Crear el modelo con el DDL del Paso 3 (Persistencia en disco)

**Nueva dinámica de persistencia:** Para no tener que recrear la base de datos cada vez que reinicias el kernel o abres el notebook, ahora verificamos si las tablas ya existen en tu MySQL local:

* **Si es la primera vez en tu PC (o si `RECREAR_TABLAS = True`):** se lee `entrega_modelo_logico.sql`, se crean la base y las 5 tablas con sus restricciones e índices.
* **Si ya existen en tu PC (por defecto `RECREAR_TABLAS = False`):** **no se borra nada**. La base de datos queda guardada en tu disco y se conservan los datos intactos.

In [ ]:
conexion = mysql.connector.connect(host=SERVIDOR, port=PUERTO, user=USUARIO, password=contrasena)
cursor = conexion.cursor()

# 1. Asegurar que la base de datos exista
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {BASE_DE_DATOS} CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;")
cursor.execute(f"USE {BASE_DE_DATOS};")

# 2. Verificar si la tabla de hechos ya fue creada
cursor.execute("SHOW TABLES LIKE 'FACT_COMPETENCIA';")
tablas_existentes = cursor.fetchone()

if not tablas_existentes or RECREAR_TABLAS:
    if RECREAR_TABLAS and tablas_existentes:
        print("RECREAR_TABLAS = True: reconstruyendo el modelo desde cero con DDL...")
    else:
        print("Primera ejecución detectada: creando estructura del Data Warehouse en MySQL...")

    with open(RUTA_DDL, encoding="utf-8") as archivo:
        texto_ddl = archivo.read()

    lineas_de_codigo = []
    for linea in texto_ddl.splitlines():
        if not linea.strip().startswith("--"):
            lineas_de_codigo.append(linea)
    texto_ddl = "\n".join(lineas_de_codigo)

    for sentencia in texto_ddl.split(";"):
        if sentencia.strip() != "":
            cursor.execute(sentencia)

    conexion.commit()
    print("Modelo y tablas creados exitosamente en", BASE_DE_DATOS)
else:
    print(f"El Data Warehouse ya existe en MySQL. Se conservan las tablas existentes (RECREAR_TABLAS={RECREAR_TABLAS}).")

cursor.close()
conexion.close()


### 5.3 Guardar las tablas (Carga con protección contra duplicados e incremental)

`DataFrame.to_sql(...)` inserta todas las filas del DataFrame en una tabla de MySQL. Con `if_exists="append"` agrega las filas a la tabla creada por el DDL.
El orden importa: **primero las dimensiones y después los hechos**, porque las claves foráneas de `FACT_COMPETENCIA` apuntan a las dimensiones.

**Nueva dinámica de inserción:**
1. **Carga inicial:** Si la tabla `FACT_COMPETENCIA` está vacía, se realiza la inserción completa de las 4 dimensiones y los hechos.
2. **Protección contra duplicados:** Si el DW ya tiene los datos cargados y no hay partidos nuevos, no se reinserta nada para evitar violaciones de clave única o duplicados.
3. **Carga incremental:** Si se agregaron filas nuevas al archivo, el script identifica las novedades, inserta solo las dimensiones nuevas y agrega únicamente los partidos que faltan.

In [ ]:
url = URL.create("mysql+mysqlconnector", username=USUARIO, password=contrasena,
                 host=SERVIDOR, port=PUERTO, database=BASE_DE_DATOS)
motor = create_engine(url)

# Verificar cuántos partidos están cargados actualmente en MySQL
filas_dw_actuales = pd.read_sql("SELECT COUNT(*) AS total FROM FACT_COMPETENCIA", motor)["total"][0]

if filas_dw_actuales == 0:
    # Caso 1: Carga inicial completa (primera vez o luego de recrear)
    print("Ejecutando carga inicial completa...")
    dim_tiempo.to_sql("DIM_TIEMPO", motor, if_exists="append", index=False)
    dim_equipo.to_sql("DIM_EQUIPO", motor, if_exists="append", index=False)
    dim_division.to_sql("DIM_DIVISION", motor, if_exists="append", index=False)
    dim_pais.to_sql("DIM_PAIS", motor, if_exists="append", index=False)
    print("Dimensiones cargadas.")

    hechos.to_sql("FACT_COMPETENCIA", motor, if_exists="append", index=False, chunksize=5000)
    print(f"Hechos cargados: {len(hechos)} partidos insertados exitosamente.")

elif filas_dw_actuales == len(hechos):
    # Caso 2: El DW ya tiene exactamente estos partidos (reinicio del kernel sin datos nuevos)
    print(f"El Data Warehouse ya está cargado ({filas_dw_actuales} partidos en FACT_COMPETENCIA).")
    print("No se reinsertaron duplicados; se preservan los datos existentes en disco.")

else:
    # Caso 3: Carga incremental (se detectaron nuevos partidos en el archivo)
    print(f"Partidos en MySQL: {filas_dw_actuales} | Partidos procesados en memoria: {len(hechos)}")
    
    # Sincronizar dimensiones nuevas (solo valores que no existan aún en MySQL)
    fechas_db = set(pd.read_sql("SELECT fecha FROM DIM_TIEMPO", motor)["fecha"].astype(str))
    equipos_db = set(pd.read_sql("SELECT nombre_equipo FROM DIM_EQUIPO", motor)["nombre_equipo"])
    divisiones_db = set(pd.read_sql("SELECT codigo_division FROM DIM_DIVISION", motor)["codigo_division"])
    paises_db = set(pd.read_sql("SELECT codigo_pais FROM DIM_PAIS", motor)["codigo_pais"])

    nuevas_fechas = dim_tiempo[~dim_tiempo["fecha"].astype(str).isin(fechas_db)]
    nuevos_equipos = dim_equipo[~dim_equipo["nombre_equipo"].isin(equipos_db)]
    nuevas_div = dim_division[~dim_division["codigo_division"].isin(divisiones_db)]
    nuevos_paises = dim_pais[~dim_pais["codigo_pais"].isin(paises_db)]

    if len(nuevas_fechas) > 0:
        nuevas_fechas.to_sql("DIM_TIEMPO", motor, if_exists="append", index=False)
        print(f"Insertadas {len(nuevas_fechas)} nuevas fechas en DIM_TIEMPO.")
    if len(nuevos_equipos) > 0:
        nuevos_equipos.to_sql("DIM_EQUIPO", motor, if_exists="append", index=False)
        print(f"Insertados {len(nuevos_equipos)} nuevos equipos en DIM_EQUIPO.")
    if len(nuevas_div) > 0:
        nuevas_div.to_sql("DIM_DIVISION", motor, if_exists="append", index=False)
        print(f"Insertadas {len(nuevas_div)} nuevas divisiones en DIM_DIVISION.")
    if len(nuevos_paises) > 0:
        nuevos_paises.to_sql("DIM_PAIS", motor, if_exists="append", index=False)
        print(f"Insertados {len(nuevos_paises)} nuevos países en DIM_PAIS.")

    # Identificar qué partidos de 'hechos' no están en MySQL todavía
    partidos_db = pd.read_sql("SELECT id_tiempo, id_equipo_local, id_equipo_visitante FROM FACT_COMPETENCIA", motor)
    partidos_nuevos = hechos.merge(
        partidos_db,
        on=["id_tiempo", "id_equipo_local", "id_equipo_visitante"],
        how="left",
        indicator=True
    )
    partidos_nuevos = partidos_nuevos[partidos_nuevos["_merge"] == "left_only"].drop(columns=["_merge"])

    if len(partidos_nuevos) > 0:
        partidos_nuevos.to_sql("FACT_COMPETENCIA", motor, if_exists="append", index=False, chunksize=5000)
        print(f"Carga incremental completada: {len(partidos_nuevos)} nuevos partidos insertados.")
    else:
        print("No se encontraron partidos nuevos para insertar.")


---
## 6. Verificación

### 6.1 Conteo de filas: MySQL contra lo que armamos

`pd.read_sql(consulta, motor)` ejecuta un `SELECT` en MySQL y devuelve el resultado como DataFrame.

In [ ]:
tablas = {
    "DIM_TIEMPO": dim_tiempo,
    "DIM_EQUIPO": dim_equipo,
    "DIM_DIVISION": dim_division,
    "DIM_PAIS": dim_pais,
    "FACT_COMPETENCIA": hechos,
}
for nombre, tabla in tablas.items():
    en_mysql = pd.read_sql("SELECT COUNT(*) AS filas FROM " + nombre, motor)["filas"][0]
    estado = "OK" if en_mysql == len(tabla) else "DIFERENCIA"
    print(nombre, "-> en MySQL:", en_mysql, "| en pandas:", len(tabla), "|", estado)

### 6.2 Balance: filas del archivo = hechos + descartes

In [ ]:
filas_hechos = pd.read_sql("SELECT COUNT(*) AS filas FROM FACT_COMPETENCIA", motor)["filas"][0]
filas_descartadas = tabla_descartes["filas"].sum()
print("Archivo:", filas_archivo)
print("Hechos:", filas_hechos)
print("Descartes:", filas_descartadas)
print("Diferencia:", filas_archivo - filas_hechos - filas_descartadas)

### 6.3 Totales del DW contra el archivo original

Volvemos a calcular los totales **directamente desde el CSV original** (la tabla `partidos`, que no tocamos), con los mismos criterios: solo ligas del alcance, solo partidos con goles y rojas ≤ 5. Y los comparamos con lo que devuelve MySQL.

In [ ]:
original = partidos[partidos["Division"].isin(mapeo_divisiones["codigo_division"])]
original = original[(original["FTHome"] != "") & (original["FTAway"] != "")]

goles_csv = pd.to_numeric(original["FTHome"]).sum() + pd.to_numeric(original["FTAway"]).sum()
rojas_csv = 0
for columna in ["HomeRed", "AwayRed"]:
    rojas = pd.to_numeric(original[columna], errors="coerce")
    rojas_csv += rojas[rojas <= 5].sum()

dw = pd.read_sql(
    "SELECT COUNT(*) AS partidos, "
    "SUM(goles_local + goles_visitante) AS goles, "
    "SUM(COALESCE(rojas_local, 0) + COALESCE(rojas_visitante, 0)) AS rojas "
    "FROM FACT_COMPETENCIA", motor)

comparacion = pd.DataFrame({
    "medida": ["partidos", "goles", "tarjetas rojas"],
    "archivo_csv": [len(original), int(goles_csv), int(rojas_csv)],
    "dw_mysql": [int(dw["partidos"][0]), int(dw["goles"][0]), int(dw["rojas"][0])],
})
comparacion["estado"] = "DIFERENCIA"
comparacion.loc[comparacion["archivo_csv"] == comparacion["dw_mysql"], "estado"] = "OK"
comparacion

### 6.4 Integridad y coherencia

* Toda clave de la tabla de hechos tiene que existir en su dimensión (las claves foráneas de MySQL ya lo garantizan; igual lo contamos).
* Cada partido tiene exactamente una de las tres banderas en 1.

In [ ]:
integridad = pd.read_sql('''
    SELECT SUM(t.id_tiempo IS NULL)   AS sin_tiempo,
           SUM(el.id_equipo IS NULL)  AS sin_equipo_local,
           SUM(ev.id_equipo IS NULL)  AS sin_equipo_visitante,
           SUM(d.id_division IS NULL) AS sin_division,
           SUM(p.id_pais IS NULL)     AS sin_pais,
           SUM(f.victoria_local + f.empate + f.victoria_visitante <> 1) AS banderas_incoherentes
    FROM FACT_COMPETENCIA f
    LEFT JOIN DIM_TIEMPO t   ON t.id_tiempo = f.id_tiempo
    LEFT JOIN DIM_EQUIPO el  ON el.id_equipo = f.id_equipo_local
    LEFT JOIN DIM_EQUIPO ev  ON ev.id_equipo = f.id_equipo_visitante
    LEFT JOIN DIM_DIVISION d ON d.id_division = f.id_division
    LEFT JOIN DIM_PAIS p     ON p.id_pais = f.id_pais
''', motor)
integridad

Si todo dio **OK** y las diferencias dan **0**, el Data Warehouse quedó cargado. El siguiente paso es abrir `02_dashboard.ipynb`.

La **política de actualización** (cada cuánto se recarga y cómo) es la misma de la versión SQL y está en `../paso-4/entrega_informe.md`, sección 7.